# This is a placeholder script for custom VOC XML to .TXT file adapter for YOLO model training

Please note that the DataTab class and structured dataset used in this script is generated and provided by the DARTIS_2019 github repository provided in the dataset's research paper on: https://github.com/yi-jie-yang/dataset_DARTIS_2019 

The repository provides building folder structure for the SAR images from Sentinal_1_DARTIS_2019_allfiles.zip and DARTIS_2019.tab (provided by the authors).

However, the data conversion of VOC XML files to YOLO-native .TXT files was not provided by the github repository. Hence, this file showcases an example script which we have used to generate our own .TXT files

In [2]:
import pandas as pd
from dataset_DARTIS_2019.dataset_toolbox import DataTab
from pathlib import Path

In [3]:
dt = DataTab("DARTIS_2019.tab")
df = dt._load_data_tab()

In [4]:
df.head()

,subset,jpg_file,xml_file,tag,patch_name,start_time,end_time,Sentinel_ID,patch_width,patch_height,...,obj_ur_lat,obj_br_lon,obj_br_lat,obj_bl_lon,obj_bl_lat,obj_patchloc_xmin,obj_patchloc_ymin,obj_patchloc_xmax,obj_patchloc_ymax,label_size
0,ow,ow-0001.jpg,ow-0001.xml,ow-0001-01-000001,S1_20190101_034235_034350_VV_1,2019-01-01T03:42:35,2019-01-01T03:43:50,S1B_IW_GRDH_1SDV_20190101T034300_20190101T0343...,640,640,...,33.263734,33.054535,33.255456,33.059407,33.254776,345.0,297.0,368.0,343.0,1058.0
1,ow,ow-0002.jpg,ow-0002.xml,ow-0002-01-000002,S1_20190104_155638_155818_VV_2,2019-01-04T15:56:38,2019-01-04T15:58:18,S1A_IW_GRDH_1SDV_20190104T155703_20190104T1557...,640,640,...,31.680507,32.031087,31.691326,32.024630,31.690382,309.0,282.0,340.0,342.0,1860.0
2,ow,ow-0003.jpg,ow-0003.xml,ow-0003-01-000003,S1_20190110_155611_155635_VV_9,2019-01-10T15:56:11,2019-01-10T15:56:35,S1B_IW_GRDH_1SDV_20190110T155611_20190110T1556...,640,640,...,31.571906,30.631705,31.575859,30.621997,31.574357,297.0,309.0,344.0,331.0,1034.0
3,ow,ow-0004.jpg,ow-0004.xml,ow-0004-01-000004,S1_20190110_155611_155635_VV_10,2019-01-10T15:56:11,2019-01-10T15:56:35,S1B_IW_GRDH_1SDV_20190110T155611_20190110T1556...,640,640,...,31.708781,31.191345,31.712193,31.188644,31.711781,347.0,292.0,360.0,311.0,247.0
4,ow,ow-0004.jpg,ow-0004.xml,ow-0004-02-000005,S1_20190110_155611_155635_VV_10,2019-01-10T15:56:11,2019-01-10T15:56:35,S1B_IW_GRDH_1SDV_20190110T155611_20190110T1556...,640,640,...,31.755418,31.237558,31.773555,31.232986,31.772857,614.0,503.0,636.0,604.0,2222.0


In [5]:
df.columns

Index(['subset', 'jpg_file', 'xml_file', 'tag', 'patch_name', 'start_time',
       'end_time', 'Sentinel_ID', 'patch_width', 'patch_height',
       'patch_ul_lon', 'patch_ul_lat', 'patch_ur_lon', 'patch_ur_lat',
       'patch_br_lon', 'patch_br_lat', 'patch_bl_lon', 'patch_bl_lat',
       'obj_ul_lon', 'obj_ul_lat', 'obj_ur_lon', 'obj_ur_lat', 'obj_br_lon',
       'obj_br_lat', 'obj_bl_lon', 'obj_bl_lat', 'obj_patchloc_xmin',
       'obj_patchloc_ymin', 'obj_patchloc_xmax', 'obj_patchloc_ymax',
       'label_size'],
      dtype='object')

In [6]:
df_oil = df[df["subset"].isin(["ow", "oc"])]
df_no_oil = df[df["subset"].isin(["nw", "nc"])]

In [7]:
df_oil_grouped = df_oil.groupby("patch_name")
df_no_oil_grouped = df_no_oil.groupby("patch_name")

In [14]:

output_label_dir = Path("labels/oil")
for patch_name, group in df_oil_grouped:
    
   # output_path = output_label_dir / (patch_name + ".txt")
    for _, row in group.iterrows():
        h = row["patch_height"]
        w = row["patch_width"]
        xmin = row["obj_patchloc_xmin"]
        xmax = row["obj_patchloc_xmax"]
        ymin = row["obj_patchloc_ymin"]
        ymax = row["obj_patchloc_ymax"]
        x_center = ((xmin + xmax) / 2.0) / w
        y_center = ((ymin + ymax) / 2.0) / h
        width = (xmax - xmin) / w
        height = (ymax - ymin) / h
        class_id = 0
        subset = row["subset"]
        if subset == "oc":
            output_path = output_label_dir / Path("coast") / (patch_name + ".txt")
        elif subset == "ow": 
            output_path = output_label_dir / Path("water") / (patch_name + ".txt")
        else:
            pass
        with open(output_path, "a") as f:
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")



In [15]:
df_oil["patch_name"].nunique()

1365

In [10]:
df_no_oil["patch_name"].nunique()

2290

In [9]:
output_label_dir = Path("labels/no_oil")
for patch_name, group in df_no_oil_grouped:
    subset = group.iloc[0]["subset"]
    if subset == "nc":
        output_path = output_label_dir / "coast" / (patch_name + ".txt")
    elif subset == "nw":
        output_path = output_label_dir / "water" / (patch_name + ".txt")
    open(output_path, "w").close()